# BreastCare AI — Resumable GPU Encoder Fine-Tuning Package (`gpu_encoder_finetune_v3`)

**Status:** Local hardware is CPU-only (`PyTorch 2.15.0+cpu`, `torch.cuda.is_available() == False`), so **full 15-fit GPU convolutional encoder fine-tuning has NOT run locally**.

This notebook is a complete, runnable, resumable Google Colab / Kaggle CUDA GPU package for [`docs/GPU_ENCODER_FINETUNE_PROTOCOL_V3.md`](../docs/GPU_ENCODER_FINETUNE_PROTOCOL_V3.md) (`ml/gpu_encoder_finetune_v2.py`):
- **Dataset & Grouping (`unified_dataset_manifest_v2.csv`):** 2,215 strictly clean eligible images across 1,288 validated duplicate/case groups (`145` Mendeley in `28` complete 5-image/10-image base-scan groups, `197` TCIA in `197` publisher patient IDs, `1,873` BUS-BRA in `1,063` source cases). All three datasets are licensed under **CC BY 4.0** and permitted for transfer to an isolated Colab/Kaggle session.
- **Architecture (`ResNet18Layer4FineTuner`):** Unfreezes `ResNet-18` `layer4` convolutional filters (`8,388,608` parameters, `lr=1e-4`) + classification head `fc` (`1,026` parameters, `lr=3e-4`; total **`8,389,634` trainable parameters**) on raw images.
- **Frozen BatchNorm Guarantee:** All 20 `nn.BatchNorm2d` modules across the entire network (including `layer4`) have `requires_grad=False` and are forced into `.eval()` mode inside `model.train()` so running statistics (`running_mean`, `running_var`) never drift.
- **Resumable Checkpoints:** Saves atomic per-epoch `.resume.pth` checkpoints and `.curve.json` learning curves under `reports/gpu_encoder_finetune_v3/models/`.

In [ ]:
# Step 1: Verify CUDA GPU hardware and repository working directory
import os, platform, torch
from pathlib import Path

print('Python:', platform.python_version())
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable a CUDA GPU runtime in Colab (Runtime -> Change runtime type -> T4 GPU) before running Step 5.'
print('GPU Device:', torch.cuda.get_device_name(0))

if Path('ml/gpu_encoder_finetune_v2.py').exists():
    REPO_ROOT = Path('.').resolve()
elif Path('../ml/gpu_encoder_finetune_v2.py').exists():
    os.chdir('..')
    REPO_ROOT = Path('.').resolve()
print('Repository root:', REPO_ROOT)

In [ ]:
# Step 2: Install/verify required Python dependencies
!pip install -q numpy scipy scikit-learn pillow matplotlib torch torchvision

In [ ]:
# Step 3: Verify CC BY 4.0 dataset paths and SHA-256 integrity across all 2,215 v2 eligible images
# Required relative paths under REPO_ROOT:
#   - data/raw/mendeley_v1/official-v1/ (CC BY 4.0, doi:10.17632/wmy84gzngw.1)
#   - data/raw/tcia_breast_biopsy_external_v1/ (CC BY 4.0, doi:10.7937/9WKK-Q141)
#   - data/research_data/busbra_reserved_v1/BUSBRA.zip (CC BY 4.0, doi:10.1002/mp.16812)
#   - reports/bounded_development_v1/preserved/pretrained_resnet18.pth
from ml.gpu_encoder_finetune_v2 import decode_pil_images, rows_v2

rs = rows_v2()
imgs = decode_pil_images(rs)
print(f'Verified all {len(imgs)} raw images and SHA-256 digests across {len({r["group"] for r in rs})} v2 groups.')

In [ ]:
# Step 4: Verify V2->V3 lock chain, frozen BatchNorm after model.train(), and layer4 gradient flow
!python -m ml.gpu_encoder_finetune_v2 verify_package

In [ ]:
# Step 5: Launch or resume the 15-fit GPU encoder fine-tuning run (gpu_encoder_finetune_v3)
# If interrupted, re-running this cell automatically resumes from the latest completed epoch (.resume.pth)
!python -m ml.gpu_encoder_finetune_v2 run --device cuda

In [ ]:
# Step 6: Export completed checkpoints, learning curves, predictions, and result bundle
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

out_root = Path('reports/gpu_encoder_finetune_v3')
bundle_path = Path('reports/gpu_encoder_finetune_v3_bundle.zip')
with ZipFile(bundle_path, 'w', ZIP_DEFLATED) as zf:
    for p in sorted(out_root.rglob('*')):
        if p.is_file() and not p.name.endswith('.resume.pth'):
            zf.write(p, str(p).replace('\\', '/'))
print('Exported GPU encoder fine-tuning bundle:', bundle_path, f'({bundle_path.stat().st_size / 1e6:.2f} MB)')